# 00b — Patch `merged_post_level.csv` to add `amp_prob_behavioral`

**Why**: `00_merged_preprocessing.ipynb` carried only `amp_prob_full` through (renamed to `amp_prob`). The behavioral label was dropped during the merge.

**Fix**: Join `tiktok_labeled.csv` back on `cid == post_id` to restore `amp_prob_behavioral`.

**Output**: `merged_post_level.csv` (overwritten) with these label columns:
- `amp_prob_full`         (renamed from `amp_prob`; TikTok only, NaN on Pantip)
- `amp_prob_behavioral`   (TikTok only, NaN on Pantip)
- `is_sockpuppet`         (Pantip only, NaN on TikTok)

In [1]:
import pandas as pd
import numpy as np

MERGED_IN  = 'merged_post_level.csv'
TT_LABELS  = 'tiktok_labeled.csv'
MERGED_OUT = 'merged_post_level.csv'  # overwrite

m = pd.read_csv(MERGED_IN)
t = pd.read_csv(TT_LABELS)

print(f'merged:        {m.shape}  cols: {list(m.columns)}')
print(f'tiktok_labels: {t.shape}  cols: {list(t.columns)}')

# Rename current single label column to its true meaning
m = m.rename(columns={'amp_prob': 'amp_prob_full'})

# Join behavioral label on post_id == cid (TikTok rows only)
m['post_id'] = m['post_id'].astype(str)
t['cid']     = t['cid'].astype(str)

m = m.merge(
    t[['cid', 'amp_prob_behavioral']].rename(columns={'cid': 'post_id'}),
    on='post_id', how='left'
)

print(f'\nAfter patch: {m.shape}  cols: {list(m.columns)}')

merged:        (25299, 11)  cols: ['source', 'post_id', 'container_id', 'user_id', 'datetime', 'text', 'length', 'likes_received', 'replies_received', 'amp_prob', 'is_sockpuppet']
tiktok_labels: (19842, 6)  cols: ['cid', 'text', 'amp_prob_full', 'amp_prob_behavioral', 'trigger_type', 'reason']

After patch: (25299, 12)  cols: ['source', 'post_id', 'container_id', 'user_id', 'datetime', 'text', 'length', 'likes_received', 'replies_received', 'amp_prob_full', 'is_sockpuppet', 'amp_prob_behavioral']


In [2]:
# ── Validation ────────────────────────────────────────────────────────────
tt = m[m['source'] == 'tiktok']
pt = m[m['source'] == 'pantip']

print('TikTok side:')
print(f'  rows:                            {len(tt):,}')
print(f'  amp_prob_full non-null:          {tt["amp_prob_full"].notna().sum():,}')
print(f'  amp_prob_behavioral non-null:    {tt["amp_prob_behavioral"].notna().sum():,}')
print(f'  full positives (>=0.5):          {(tt["amp_prob_full"]>=0.5).sum():,}')
print(f'  behavioral positives (>=0.5):    {(tt["amp_prob_behavioral"]>=0.5).sum():,}')
print()
print('Pantip side:')
print(f'  rows:                            {len(pt):,}')
print(f'  is_sockpuppet non-null:          {pt["is_sockpuppet"].notna().sum():,}')
print(f'  sockpuppet positives:            {(pt["is_sockpuppet"]==1).sum():,}')
print()
# Expected: TT full=4498, TT behavioral=1972
assert (tt['amp_prob_full']>=0.5).sum() == 4498, 'full positives mismatch'
assert (tt['amp_prob_behavioral']>=0.5).sum() == 1972, 'behavioral positives mismatch'
print('✓ All expected positive counts match.')

TikTok side:
  rows:                            19,842
  amp_prob_full non-null:          19,842
  amp_prob_behavioral non-null:    19,842
  full positives (>=0.5):          4,498
  behavioral positives (>=0.5):    1,972

Pantip side:
  rows:                            5,457
  is_sockpuppet non-null:          5,457
  sockpuppet positives:            2,390

✓ All expected positive counts match.


In [3]:
m.to_csv(MERGED_OUT, index=False)
print(f'Saved → {MERGED_OUT}')
print(f'  Rows: {len(m):,}')
print(f'  Cols: {list(m.columns)}')

Saved → merged_post_level.csv
  Rows: 25,299
  Cols: ['source', 'post_id', 'container_id', 'user_id', 'datetime', 'text', 'length', 'likes_received', 'replies_received', 'amp_prob_full', 'is_sockpuppet', 'amp_prob_behavioral']
